# Data audit and cohort derivation

**Study.** Incremental Predictive Value of Two Antenatal Visits for Recorded Low Birth Weight: A Retrospective Cohort Study. Kelvin Rwihimba, BSc Software Engineering, African Leadership University. Supervisor: Emmanuel Adjei.

This notebook is a narrative over the `antenatal` package in `src/`. It does no calculation of its own: every table below is produced by a tested package function, so the numbers here are the numbers the pipeline, the report and the prototype use.

It answers four questions in order.

1. Is this exactly the released file, and what does it contain?
2. What is wrong with the file, and how is each defect treated?
3. Which pregnancies enter the study, and why are the others left out?
4. Which records are sealed away for the single final check, and what does the missingness in the remaining records look like?

Run it from the `notebooks` folder after `make install`. It needs no input.

In [1]:
from pathlib import Path

import pandas as pd

from antenatal.audit.missingness import missingness_mechanism, missingness_table
from antenatal.audit.quality import (
    consistency_checks,
    non_numeric_tokens,
    quality_findings,
    registration_ranges,
    suspect_field_crosstabs,
)
from antenatal.cohort.builder import CohortBuilder
from antenatal.cohort.partition import PartitionWriter
from antenatal.config import PRIMARY, sensitivity_specs
from antenatal.data.loader import DataLoader
from antenatal.paths import ProjectPaths

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)
paths = ProjectPaths.from_root(Path.cwd().parent)
print("Repository root:", paths.root.name)
print("Seed:", PRIMARY.seed)

Repository root: Antanental_risk_model
Seed: 20260930


## 1. Provenance

The study uses version 1 of the Dataset for Maternal Health Risks Stratification (MHRS) in Tanzania (Mwifunyi et al., 2025), released under a Creative Commons Attribution 4.0 licence. The loader checks the file's SHA 256 fingerprint before reading it. If one byte differs, it stops with `ChecksumMismatch`, so every result is tied to one exact file. The file is opened for reading only and is never written to.

In [2]:
loader = DataLoader(PRIMARY, paths.root)
raw = loader.load()
pd.Series(loader.provenance()).to_frame("value")

,value
dataset,Dataset for Maternal Health Risks Stratification (MHRS) in Tanzania
version,1
doi,10.5281/zenodo.15309733
licence,CC BY 4.0
file,data/maternal_dataset_csv.csv
bytes,15074677
sha256,ab95366ef22dde5d13df93f10723942f5faffda8fbf0007e8efe85e4f25c1126
rows,8817
columns,683


## 2. Type and sentinel audit

Many fields that should hold numbers also hold words such as `not_checked` or `invalid`. These words mean "no measurement", so they must become missing values and never zeros. The table counts every such token in columns that are mostly numeric. `treated_as_missing` shows that the token is in the study's sentinel list. `repaired_as_number` marks typing slips such as `36.5.` that the numeric parser repairs because their meaning is clear.

In [3]:
tokens = non_numeric_tokens(raw)
summary = tokens.groupby("token")["rows"].sum().sort_values(ascending=False)
print(f"{tokens['column'].nunique()} mostly numeric text columns hold {len(summary)} distinct tokens")
print(f"Rows holding a sentinel token: {int(tokens.loc[tokens['treated_as_missing'], 'rows'].sum()):,}")
summary.head(12).to_frame("rows")

32 mostly numeric text columns hold 65 distinct tokens
Rows holding a sentinel token: 21,498


,rows
token,
not_checked,19313
normal,1983
not checked,1631
nomal,1043
no,542
negative,458
36.5.,16
36.6.,13
36.0.,7


## 3. Data quality audit (Table 3.3)

Three history fields agree with delivery outcome fields far more often than real history should. The agreement is consistent with the outcome having been copied into the history section, but the publisher has not confirmed this, so the fields are called suspect rather than duplicates. All three, and the miscarriage count, are excluded from the predictors. These checks describe the file itself and use no birth weight outcome, so they cannot leak information from the held out records into any modelling decision.

In [4]:
quality_findings(raw)

,finding,count,denominator
0,preterm_labour_1 agrees with preterm_labour,8808.0,8817
1,labour_at_term_1 agrees with labour_at_term,8530.0,8817
2,miscarriages_or_abortions positive where miscarriages_or_abortions_1 is positive,84.0,86
3,Records marked as both term delivery and preterm labour,157.0,8817
4,Recorded birth weight above 6 kg,35.0,8817
5,"Maximum recorded birth weight, kg",500.0,8817
6,Exact duplicate rows ignoring the sequence number,1.0,8817


In [5]:
for name, table in suspect_field_crosstabs(raw).items():
    print(name)
    print(table.to_string())
    print()

preterm_labour_1 by preterm_labour
preterm_labour      no  yes
preterm_labour_1           
no                8459    9
yes                  0  349

labour_at_term_1 by labour_at_term
labour_at_term     no   yes
labour_at_term_1           
no                397   287
yes                 0  8133

miscarriages_or_abortions by miscarriages_or_abortions_1
miscarriages_or_abortions_1    no  yes
miscarriages_or_abortions             
no                           8538    2
yes                           193   84



Registration values outside physiological ranges, such as an age of 121 or a height of 0 cm, are set to missing by the range rules in `StudySpec.ranges`. The record is kept. The observed extremes before cleaning are below.

In [6]:
registration_ranges(raw)

,field,minimum,maximum
0,age,11.0,121.0
1,number of pregnancies,1.0,53.0
2,"height, cm",0.0,180.0
3,"registration weight, kg",1.0,11080.0


Related fields should tell the same story. The first check shows that the parity fields are unreliable. The second shows that the dates disagree with each other, so no dates are used. The third shows that two different visit one questions hold identical answers in every row, which points to an export error. Neither of those two visit one fields enters any feature set.

In [7]:
consistency_checks(raw)

,check,count,denominator
0,Prior deliveries at or above number of pregnancies,4881,8817
1,Expected delivery 275 to 285 days after last menstrual date,2066,8811
2,Visit one violence answer identical to visit one substance answer,8817,8817


## 4. Cohort derivation (Table 3.5)

`CohortBuilder` applies one named method per rule, in the order of Table 3.5, and records for every record the step and the reason that removed it. The first three steps define the eligible cohort. The last four require two usable visits, the second later in pregnancy than the first and at or before 28 weeks. The per record file is `outputs/cohort_manifest.csv`.

In [8]:
cohort = CohortBuilder(PRIMARY).build(raw)
flow = cohort.flow_table()
flow

,number,criterion,excluded,remaining,events_remaining
0,0,Records in the file,0,8817,443
1,1,No recorded live birth,208,8609,382
2,2,Multiple delivery,416,8193,360
3,3,Recorded birth weight outside 1 to 6 kilograms,32,8161,360
4,4,Visit one lacks a parsed blood pressure in range or a gestational age in range,167,7994,349
5,5,"Visit two absent, or lacks a parsed blood pressure in range or a gestational age in range",1791,6203,271
6,6,Gestational age at visit two not greater than at visit one,138,6065,268
7,7,Visit two later than 28 weeks,2157,3908,206


In [9]:
eligible = cohort.records[cohort.records["eligible"]]
print(f"Eligible cohort: {len(eligible):,} pregnancies, {int(eligible['lbw'].sum())} low birth weight "
      f"({100 * eligible['lbw'].mean():.2f} percent)")
print(f"Analysis cohort: {len(cohort.analysis):,} pregnancies, {cohort.events} low birth weight "
      f"({100 * cohort.events / len(cohort.analysis):.2f} percent)")
largest = flow.iloc[1:].sort_values("excluded", ascending=False).iloc[0]
print(f"Largest single exclusion: step {largest['number']}, {largest['criterion']} ({largest['excluded']:,})")
bp_rate = cohort.records.loc[cohort.records['eligible'], 'bp_parsed_v1'].mean()
print(f"Visit one blood pressure parse rate in the eligible cohort: {100 * bp_rate:.1f} percent")

Eligible cohort: 8,161 pregnancies, 360 low birth weight (4.41 percent)
Analysis cohort: 3,908 pregnancies, 206 low birth weight (5.27 percent)
Largest single exclusion: step 7, Visit two later than 28 weeks (2,157)
Visit one blood pressure parse rate in the eligible cohort: 98.2 percent


The sensitivity cohorts of Table 3.8 come from the same builder with one field of the specification changed. S1 and S2 move the gestational cutoff, S3 removes records marked as both term and preterm, and S6 relaxes the lower birth weight bound to 0.5 kg.

In [10]:
rows = [{"analysis": "Primary", "cohort": len(cohort.analysis), "events": cohort.events}]
for name, spec in sensitivity_specs().items():
    variant = CohortBuilder(spec).build(raw)
    rows.append({"analysis": name, "cohort": len(variant.analysis), "events": variant.events})
sensitivity = pd.DataFrame(rows)
sensitivity["prevalence_percent"] = (100 * sensitivity["events"] / sensitivity["cohort"]).round(2)
sensitivity

,analysis,cohort,events,prevalence_percent
0,Primary,3908,206,5.27
1,S1,2449,142,5.80
2,S2,5083,240,4.72
3,S3,3848,201,5.22
4,S6,3908,206,5.27


## 5. The sealed partition

Before any availability screen, model choice or cross validation step, 20 percent of the analysis cohort is sealed. Records that share a registration profile (age, last menstrual date, height, registration weight and prior deliveries) stay on the same side, and the split keeps the outcome rate balanced. The identifiers are written to `partitions/holdout_ids.csv` with a SHA 256, and the writer refuses to overwrite them. This notebook reads only the development records. The held out records can be opened only through `HoldoutEvaluator`, which logs every attempt.

In [11]:
writer = PartitionWriter(paths.partitions, PRIMARY)
partition = writer.load()
development = partition.development(cohort)
print(f"Partition SHA 256: {partition.sha256}")
print(f"Development partition: {len(development):,} pregnancies, {int(development['lbw'].sum())} events")
print(f"Held out partition: {len(partition.holdout_ids):,} pregnancies (outcome not read here)")
shared = development["group_key"].duplicated(keep=False).sum()
print(f"Development records sharing a registration profile with another record: {shared}")

Partition SHA 256: 249114c363dab8a86c5acbee9bb8cfd6fddbf97b3d74b62f907d1fda01a25ea0
Development partition: 3,126 pregnancies, 165 events
Held out partition: 782 pregnancies (outcome not read here)
Development records sharing a registration profile with another record: 4


## 6. Missingness in the development records

Missingness is measured before any treatment. Blood pressure and gestational age are complete by construction, because the cohort rules require them. Pulse and haemoglobin are often missing at both visits.

In [12]:
missingness_table(development).pivot(index="measurement", columns="visit", values="percent_missing")

visit,1,2
measurement,,
dbp,0.0,0.0
ga,0.0,0.0
hb,9.6,61.2
pulse,18.6,18.6
sbp,0.0,0.0
weight,0.2,0.3


Is missingness related to the outcome? The table compares the low birth weight rate between records with and without a visit two value, with Wilson 95 percent intervals. The last column shows how often the value is present across tenths of the file in record order. Record order is the only proxy for time, because the file has no visit dates. If the rates are close and presence drifts with record order, missingness looks driven by clinic practice rather than by the outcome. That is the condition under which median imputation with a missingness indicator, the treatment the proposal specifies, is reasonable.

In [13]:
missingness_mechanism(development)

,measurement,group,n,events,lbw_percent,ci_low_percent,ci_high_percent,percent_present_by_record_decile
0,hb at visit two,value present,1213,61,5.03,3.93,6.41,"[38, 39, 37, 38, 37, 46, 39, 36, 40, 38]"
1,hb at visit two,value missing,1913,104,5.44,4.51,6.54,"[38, 39, 37, 38, 37, 46, 39, 36, 40, 38]"
2,pulse at visit two,value present,2544,146,5.74,4.90,6.71,"[88, 81, 80, 81, 83, 77, 79, 88, 83, 73]"
3,pulse at visit two,value missing,582,19,3.26,2.10,5.04,"[88, 81, 80, 81, 83, 77, 79, 88, 83, 73]"


In [14]:
check = loader.verify_checksum()
print("Raw file unchanged after the audit:", check == PRIMARY.data_sha256)

Raw file unchanged after the audit: True


## Summary

The released file was verified by checksum and audited without being modified. The audit found suspect history fields, impossible registration values, inconsistent parity and date fields, and a duplicated pair of visit one questions; each finding has a stated treatment. The cohort rule of Table 3.5 was applied by one tested builder, and the counts above are the counts every later step uses. The held out partition was sealed before any modelling step, and every outcome based summary in this notebook uses development records only.

Reference: Mwifunyi, R., et al. (2025). Dataset for maternal health risks stratification (MHRS) in Tanzania (Version 1). Zenodo. https://doi.org/10.5281/zenodo.15309733